In [8]:
import sys
from pathlib import Path
import numpy as np

# =========================
# 0) 路径配置：改这里
# =========================
# 选择后端: "pytorch" / "qonnx" / "finn"
BACKEND = "finn"

# ECG5000 测试集
DATA_PATH = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/datasets/ECG5000/ECG5000_TEST.txt")

# 你的 pytorch-tcn 工程根目录（为了 import 模型类）
PROJECT_DIR = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn")

# PyTorch checkpoint
CKPT_PATH = PROJECT_DIR / "tests" / "best_qtcn_model.pth"

# QONNX / FINN 模型
QONNX_PATH = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_qonnx_clean.onnx")
FINN_PATH  = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn.onnx")

# 如果只想先测前 N 条，就写数字；全测写 None
MAX_SAMPLES = None

# 如果某个 ONNX 输入 datatype 是 UINT8，可用这个固定 scale
INPUT_SCALE = 0.041415304

# 打印前几条预测
PRINT_FIRST_N = 10


# =========================
# 1) 通用工具函数
# =========================
def load_ecg5000_txt(path: Path):
    arr = np.loadtxt(path, dtype=np.float32)
    y = arr[:, 0].astype(np.int32)
    x = arr[:, 1:].astype(np.float32)

    uniq = np.unique(y)
    if np.array_equal(uniq, np.array([1, 2, 3, 4, 5])):
        y = y - 1

    return x, y


def confusion_matrix_np(y_true, y_pred, num_classes=5):
    cm = np.zeros((num_classes, num_classes), dtype=np.int32)
    for t, p in zip(y_true, y_pred):
        if 0 <= t < num_classes and 0 <= p < num_classes:
            cm[t, p] += 1
    return cm


def print_summary(y_true, y_pred, title="Result"):
    y_true = np.asarray(y_true, dtype=np.int32)
    y_pred = np.asarray(y_pred, dtype=np.int32)

    acc = float(np.mean(y_true == y_pred))
    cm = confusion_matrix_np(y_true, y_pred, num_classes=5)

    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    print(f"Samples   : {len(y_true)}")
    print(f"Accuracy  : {acc:.4f}")
    print("Confusion matrix:")
    print(cm)

    print("\nPer-class accuracy:")
    for c in range(5):
        total_c = np.sum(y_true == c)
        correct_c = np.sum((y_true == c) & (y_pred == c))
        acc_c = correct_c / total_c if total_c > 0 else 0.0
        print(f"class {c}: {correct_c}/{total_c} = {acc_c:.4f}")


def encode_u8_fixed_scale(x_float: np.ndarray, scale: float = INPUT_SCALE):
    x = np.asarray(x_float, dtype=np.float32).reshape(-1)
    q = np.round(x / scale)
    q = np.clip(q, 0, 255).astype(np.uint8)
    return q


# =========================
# 2) PyTorch 后端
# =========================
def eval_pytorch(X, Y):
    import torch

    sys.path.insert(0, str(PROJECT_DIR))
    from quant_tcn.ECG5000_QTCN import ECG5000QTCNClassifier

    device = torch.device("cpu")

    model = ECG5000QTCNClassifier(num_classes=5).to(device)
    ckpt = torch.load(CKPT_PATH, map_location=device)

    # 兼容两种 checkpoint 格式
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
    else:
        state_dict = ckpt

    model.load_state_dict(state_dict, strict=True)
    model.eval()

    preds = []

    with torch.no_grad():
        for i in range(len(X)):
            x = torch.tensor(X[i], dtype=torch.float32).view(1, 1, 140).to(device)
            logits = model(x).cpu().numpy().reshape(-1)
            pred = int(np.argmax(logits))
            preds.append(pred)

            if i < PRINT_FIRST_N:
                print(f"[{i}] y_true={int(Y[i])}, pred={pred}, logits={np.round(logits, 4)}")

            if (i + 1) % 100 == 0:
                acc_so_far = np.mean(np.array(preds) == Y[:len(preds)])
                print(f"Processed {i+1}/{len(X)}, acc_so_far={acc_so_far:.4f}")

    return np.asarray(preds, dtype=np.int32)


# =========================
# 3) ONNX/QONNX/FINN 后端
# =========================
def eval_onnx(X, Y, model_path: Path, title="ONNX"):
    from qonnx.core.modelwrapper import ModelWrapper
    import qonnx.core.onnx_exec as oxe

    model = ModelWrapper(str(model_path))

    iname = model.graph.input[0].name
    oname = model.graph.output[0].name
    ishape = model.get_tensor_shape(iname)
    idtype = model.get_tensor_datatype(iname)

    print("Model path :", model_path)
    print("Input name :", iname)
    print("Input shape:", ishape)
    print("Input dtype:", idtype)
    print("Output name:", oname)

    preds = []

    for i in range(len(X)):
        x = X[i].astype(np.float32).reshape(1, 1, 140)

        # 根据模型输入 datatype 决定怎么喂
        if str(idtype) == "FLOAT32":
            inp = x.astype(np.float32)

        elif str(idtype) == "UINT8":
            inp = encode_u8_fixed_scale(X[i]).reshape(1, 1, 140)

        elif str(idtype) == "INT8":
            # 这里只给一个保底写法；你当前重点不在这个分支
            q = np.round(X[i] / INPUT_SCALE)
            q = np.clip(q, -128, 127).astype(np.int8)
            inp = q.reshape(1, 1, 140)

        else:
            raise RuntimeError(f"Unsupported input datatype: {idtype}")

        out_dict = oxe.execute_onnx(model, {iname: inp})
        logits = np.asarray(out_dict[oname]).reshape(-1)
        pred = int(np.argmax(logits))
        preds.append(pred)

        if i < PRINT_FIRST_N:
            print(f"[{i}] y_true={int(Y[i])}, pred={pred}, logits={logits}")

        if (i + 1) % 100 == 0:
            acc_so_far = np.mean(np.array(preds) == Y[:len(preds)])
            print(f"Processed {i+1}/{len(X)}, acc_so_far={acc_so_far:.4f}")

    return np.asarray(preds, dtype=np.int32)


# =========================
# 4) 主程序
# =========================
def main():
    X, Y = load_ecg5000_txt(DATA_PATH)

    if MAX_SAMPLES is not None:
        X = X[:MAX_SAMPLES]
        Y = Y[:MAX_SAMPLES]

    print("Loaded dataset:")
    print("X shape =", X.shape)
    print("Y shape =", Y.shape)
    print("Label set =", np.unique(Y))

    if BACKEND == "pytorch":
        preds = eval_pytorch(X, Y)
        print_summary(Y, preds, title="PyTorch checkpoint result")

    elif BACKEND == "qonnx":
        preds = eval_onnx(X, Y, QONNX_PATH, title="QONNX")
        print_summary(Y, preds, title="QONNX result")

    elif BACKEND == "finn":
        preds = eval_onnx(X, Y, FINN_PATH, title="FINN ONNX")
        print_summary(Y, preds, title="FINN ONNX result")

    else:
        raise ValueError(f"Unknown BACKEND: {BACKEND}")


if __name__ == "__main__":
    main()

Loaded dataset:
X shape = (4500, 140)
Y shape = (4500,)
Label set = [0 1 2 3 4]
Model path : /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn.onnx
Input name : global_in
Input shape: [1, 1, 140]
Input dtype: FLOAT32
Output name: global_out


/home/slowman/Desktop/project/Thesis/finn/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_0_out0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(
/home/slowman/Desktop/project/Thesis/finn/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_11_out0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(


[0] y_true=0, pred=0, logits=[ 2.2998397  1.4347253 -2.355272  -0.9034588 -3.9731069]
[1] y_true=0, pred=0, logits=[ 2.5108738  1.3637321 -2.2431562 -1.1768721 -3.9671335]
[2] y_true=0, pred=0, logits=[ 2.8204088  1.4511192 -2.5294898 -1.3419204 -4.4481077]
[3] y_true=0, pred=0, logits=[ 2.6670303  1.4898803 -2.4312668 -1.1820124 -4.293757 ]
[4] y_true=0, pred=0, logits=[ 2.622573   1.3019087 -2.2703867 -1.2082702 -4.089113 ]
[5] y_true=0, pred=0, logits=[ 2.74497    1.3731796 -2.366109  -1.2819028 -4.355441 ]
[6] y_true=0, pred=0, logits=[ 1.8681853  1.4541754 -2.0065594 -0.7852297 -3.323055 ]
[7] y_true=0, pred=0, logits=[ 2.4232092  1.4029105 -2.2441287 -1.0967097 -3.9251764]
[8] y_true=0, pred=0, logits=[ 2.2849743  1.2725946 -2.0351791 -1.0698963 -3.6228654]
[9] y_true=0, pred=0, logits=[ 2.9278016  1.6065813 -2.7551117 -1.3822099 -4.7352743]
Processed 100/4500, acc_so_far=1.0000
Processed 200/4500, acc_so_far=1.0000
Processed 300/4500, acc_so_far=1.0000
Processed 400/4500, acc_so

In [9]:
from qonnx.core.modelwrapper import ModelWrapper

MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_create_dataflow_partition.onnx"

m = ModelWrapper(MODEL_PATH)

for n in m.graph.node:
    if n.op_type == "StreamingDataflowPartition":
        print("Partition node name:", n.name)
        print("Inputs :", list(n.input))
        print("Outputs:", list(n.output))
        print()

        for t in list(n.input) + list(n.output):
            try:
                shp = m.get_tensor_shape(t)
            except:
                shp = None
            try:
                dt = m.get_tensor_datatype(t)
            except:
                dt = None
            print(f"{t}: shape={shp}, dtype={dt}")

In [10]:
from qonnx.core.modelwrapper import ModelWrapper
from collections import Counter
from pathlib import Path

MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_create_dataflow_partition.onnx"

p = Path(MODEL_PATH)
print("exists:", p.exists())
print("path  :", p)

m = ModelWrapper(str(p))

print("graph inputs :")
for x in m.graph.input:
    print(" -", x.name)

print("\ngraph outputs:")
for x in m.graph.output:
    print(" -", x.name)

ops = Counter([n.op_type for n in m.graph.node])

print("\nnode count:", len(m.graph.node))
print("\nop_type summary:")
for k, v in ops.most_common():
    print(f"{k}: {v}")

exists: True
path  : /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_create_dataflow_partition.onnx
graph inputs :
 - MultiThreshold_11_out0

graph outputs:
 - MatMul_7_out0

node count: 1

op_type summary:
MVAU: 1


In [11]:
from qonnx.core.modelwrapper import ModelWrapper
from pathlib import Path

MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_create_dataflow_partition.onnx"

m = ModelWrapper(MODEL_PATH)

hits = []
for n in m.graph.node:
    text = f"{n.name} {n.op_type} {n.domain}".lower()
    if any(k in text for k in ["partition", "stream", "dataflow", "finn"]):
        hits.append(n)

print("matched nodes:", len(hits))
for i, n in enumerate(hits):
    print("=" * 60)
    print(f"[{i}]")
    print("name   :", n.name)
    print("op_type:", n.op_type)
    print("domain :", n.domain)
    print("inputs :", list(n.input))
    print("outputs:", list(n.output))

matched nodes: 1
[0]
name   : MVAU_0
op_type: MVAU
domain : finn.custom_op.fpgadataflow
inputs : ['MultiThreshold_11_out0', 'MatMul_7_param0']
outputs: ['MatMul_7_out0']


In [2]:
import numpy as np
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe

MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/onnx_exports/ecg5000_static_qtcn_finn.onnx"
DATA_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/datasets/ECG5000/ECG5000_TEST.txt"

m = ModelWrapper(MODEL_PATH)

iname = m.graph.input[0].name
oname = m.graph.output[0].name

print("model input :", iname)
print("model output:", oname)

arr = np.loadtxt(DATA_PATH, dtype=np.float32)
Y = arr[:, 0].astype(np.int32)
X = arr[:, 1:].astype(np.float32)

if np.array_equal(np.unique(Y), np.array([1,2,3,4,5])):
    Y = Y - 1

# 先取一条样本
idx = 0
x = X[idx].reshape(1, 1, 140).astype(np.float32)

out_dict = oxe.execute_onnx(m, {iname: x}, return_full_exec_context=True)

print("y_true =", Y[idx])
print("final logits =", out_dict[oname])

mid = out_dict["MultiThreshold_11_out0"]
print("MultiThreshold_11_out0 shape:", mid.shape)
print("MultiThreshold_11_out0 dtype:", mid.dtype)
print("MultiThreshold_11_out0:", mid)

model input : global_in
model output: global_out


/home/slowman/Desktop/project/Thesis/finn/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_0_out0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(


y_true = 0
final logits = [[ 2.2998397  1.4347253 -2.355272  -0.9034588 -3.9731069]]
MultiThreshold_11_out0 shape: (1, 16)
MultiThreshold_11_out0 dtype: float32
MultiThreshold_11_out0: [[234. 190. 196. 130. 170. 133. 160. 161. 137. 212. 166. 148. 176. 237.
  213. 181.]]


/home/slowman/Desktop/project/Thesis/finn/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_11_out0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(


In [3]:
from qonnx.core.modelwrapper import ModelWrapper
from collections import Counter

MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_convert_to_hw.onnx"

m = ModelWrapper(MODEL_PATH)

print("graph inputs:")
for x in m.graph.input:
    print(" -", x.name)

print("\ngraph outputs:")
for x in m.graph.output:
    print(" -", x.name)

domain_counter = Counter()
op_counter = Counter()

for n in m.graph.node:
    domain_counter[n.domain] += 1
    op_counter[(n.domain, n.op_type)] += 1

print("\nDomain summary:")
for k, v in domain_counter.items():
    print(k, ":", v)

print("\nNodes in finn.custom_op.fpgadataflow:")
for n in m.graph.node:
    if n.domain == "finn.custom_op.fpgadataflow":
        print("=" * 60)
        print("name   :", n.name)
        print("op_type:", n.op_type)
        print("inputs :", list(n.input))
        print("outputs:", list(n.output))

graph inputs:
 - global_in

graph outputs:
 - global_out

Domain summary:
 : 48
qonnx.custom_op.general : 18
finn.custom_op.fpgadataflow : 1

Nodes in finn.custom_op.fpgadataflow:
name   : MVAU_0
op_type: MVAU
inputs : ['MultiThreshold_11_out0', 'MatMul_7_param0']
outputs: ['MatMul_7_out0']


In [4]:
from qonnx.core.modelwrapper import ModelWrapper

MODEL_PATH = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_convert_to_hw.onnx"

m = ModelWrapper(MODEL_PATH)

for i, n in enumerate(m.graph.node):
    if n.domain != "finn.custom_op.fpgadataflow":
        print("=" * 60)
        print(f"[{i}]")
        print("name   :", n.name)
        print("op_type:", n.op_type)
        print("domain :", n.domain)
        print("inputs :", list(n.input))
        print("outputs:", list(n.output))

[0]
name   : Identity_0
op_type: Identity
domain : 
inputs : ['Concat_4_param0']
outputs: ['Identity_0_out0']
[1]
name   : Identity_1
op_type: Identity
domain : 
inputs : ['Concat_2_param0']
outputs: ['Identity_1_out0']
[2]
name   : Identity_2
op_type: Identity
domain : 
inputs : ['Concat_0_param0']
outputs: ['Identity_2_out0']
[3]
name   : MultiThreshold_0
op_type: MultiThreshold
domain : qonnx.custom_op.general
inputs : ['global_in', 'MultiThreshold_0_param0']
outputs: ['MultiThreshold_0_out0']
[4]
name   : Mul_0
op_type: Mul
domain : 
inputs : ['MultiThreshold_0_out0', 'Mul_0_param0']
outputs: ['Mul_0_out0']
[5]
name   : Unsqueeze_0
op_type: Unsqueeze
domain : 
inputs : ['Mul_0_out0', 'Unsqueeze_0_param0']
outputs: ['Unsqueeze_0_out0']
[6]
name   : Transpose_0
op_type: Transpose
domain : 
inputs : ['Unsqueeze_0_out0']
outputs: ['Transpose_0_out0']
[7]
name   : MatMul_0
op_type: MatMul
domain : 
inputs : ['Transpose_0_out0', 'MatMul_0_param0']
outputs: ['MatMul_0_out0']
[8]
name   : 